# Modelado T+3 — Tasas ponderadas de crédito de consumo

Este notebook toma las bases curadas del ETL y construye un sistema de predicción jerárquico para estimar la **tasa ponderada tres meses adelante**.

## Estrategia

1. Intentar predicción granular por `banco + rango_monto`.
2. Si una serie no es apta, usar fallback por `banco`.
3. Si el banco no es apto, usar fallback por `rango_monto`.
4. Si nada anterior aplica, usar el modelo agregado del sistema.

La métrica principal de selección es `R2`, sin perder de vista `MAE`, `RMSE` y `sMAPE`. Todos los modelos optimizados se comparan contra baselines simples.

**Documentación (Markdown):** [README.md](README.md) · [Informe modelado por sección](docs/INFORME_MODELADO.md)

## 1. Configuración

Los parámetros de esta sección controlan horizonte, validación temporal y presupuesto de optimización bayesiana. El presupuesto está configurado como intensivo, pero puede bajarse durante pruebas rápidas.

In [3]:
import sys
!{sys.executable} -m pip install optuna xgboost joblib scikit-learn pandas numpy

In [5]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import joblib
import optuna
from optuna.samplers import TPESampler

try:
    from xgboost import XGBRegressor
    XGBOOST_DISPONIBLE = True
except Exception as exc:
    XGBOOST_DISPONIBLE = False
    print(f"XGBoost no disponible: {exc}")

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 80)

RANDOM_STATE = 42
HORIZONTE_MESES = 3
OUTPUT_ETL = Path("outputs")
OUTPUT_MODELOS = Path("outputs_modelado")
OUTPUT_MODELOS.mkdir(exist_ok=True)

CONFIG = {
    "min_meses_serie": 18,
    "min_observaciones_target": 12,
    "min_creditos_serie": 100,
    "min_cobertura_meses": 0.80,
    "test_meses": 6,
    "valid_meses": 6,
    "n_trials_intensivo": 80,
    "timeout_estudio_segundos": None,
    "n_jobs_modelos_arboles": -1,
    "r2_minimo_aceptable": 0.0,
}

print("Configuración de modelado cargada")
print(f"XGBoost disponible: {XGBOOST_DISPONIBLE}")
print(f"Directorio ETL: {OUTPUT_ETL.resolve()}")
print(f"Directorio modelado: {OUTPUT_MODELOS.resolve()}")

Configuración de modelado cargada
XGBoost disponible: True
Directorio ETL: C:\Users\CAMILO\Aprendizage automatico\Juan Camilos 2\outputs
Directorio modelado: C:\Users\CAMILO\Aprendizage automatico\Juan Camilos 2\outputs_modelado


## 2. Carga de datos del ETL

Se cargan las tasas ponderadas mensuales exportadas por el ETL. A partir de `banco-rango-mes` se reconstruye también el nivel `rango-mes`, que funcionará como fallback intermedio.

In [8]:
ARCHIVOS_REQUERIDOS = {
    "banco_rango_mes": OUTPUT_ETL / "tasas_ponderadas_banco_rango_mes.csv",
    "banco_mes": OUTPUT_ETL / "tasas_ponderadas_banco_mes.csv",
    "total_mes": OUTPUT_ETL / "tasas_ponderadas_total_mes.csv",
    "aptitud_t2": OUTPUT_ETL / "resumen_aptitud_banco_rango.csv",
    "continuidad_bancos": OUTPUT_ETL / "continuidad_bancos.csv",
}

faltantes = [nombre for nombre, ruta in ARCHIVOS_REQUERIDOS.items() if not ruta.exists()]
if faltantes:
    raise FileNotFoundError(f"Faltan archivos del ETL: {faltantes}")

banco_rango_mes = pd.read_csv(ARCHIVOS_REQUERIDOS["banco_rango_mes"])
banco_mes = pd.read_csv(ARCHIVOS_REQUERIDOS["banco_mes"])
total_mes = pd.read_csv(ARCHIVOS_REQUERIDOS["total_mes"])
aptitud_t2 = pd.read_csv(ARCHIVOS_REQUERIDOS["aptitud_t2"])
continuidad_bancos = pd.read_csv(ARCHIVOS_REQUERIDOS["continuidad_bancos"])

for df in [banco_rango_mes, banco_mes, total_mes]:
    df["mes_periodo"] = pd.PeriodIndex(df["mes"], freq="M")

rango_mes = (
    banco_rango_mes.groupby(["mes", "mes_periodo", "rango_orden", "rango_monto"], dropna=False)
    .agg(
        suma_tasa_credito=("suma_tasa_credito", "sum"),
        total_creditos=("total_creditos", "sum"),
        registros_fuente=("registros_fuente", "sum"),
        fecha_minima=("fecha_minima", "min"),
        fecha_maxima=("fecha_maxima", "max"),
        bancos=("banco", "nunique"),
    )
    .reset_index()
)
rango_mes["tasa_ponderada"] = rango_mes["suma_tasa_credito"] / rango_mes["total_creditos"]

print("banco_rango_mes:", banco_rango_mes.shape)
print("banco_mes      :", banco_mes.shape)
print("rango_mes      :", rango_mes.shape)
print("total_mes      :", total_mes.shape)
print("continuidad bancos:", continuidad_bancos.shape)

display(banco_rango_mes.head())
display(rango_mes.head())

banco_rango_mes: (3591, 11)
banco_mes      : (696, 9)
rango_mes      : (186, 11)
total_mes      : (31, 8)
continuidad bancos: (26, 6)


,mes,banco,rango_orden,rango_monto,suma_tasa_credito,total_creditos,registros_fuente,fecha_minima,fecha_maxima,tasa_ponderada,mes_periodo
0,2023-09,BBVA Colombia,1,Menor o igual a 3 SMLMV,13163.74,385,189,2023-09-29,2023-09-29,34.191532,2023-09
1,2023-09,BBVA Colombia,2,Mayor a 3 SMLMV y menor o igual a 6 SMLMV,19855.53,578,244,2023-09-29,2023-09-29,34.352128,2023-09
2,2023-09,BBVA Colombia,3,Mayor a 6 SMLMV y menor o igual a 12 SMLMV,26798.80,805,307,2023-09-29,2023-09-29,33.290435,2023-09
3,2023-09,BBVA Colombia,4,Mayor a 12 SMLMV y menor o igual a 25 SMLMV,22025.51,679,278,2023-09-29,2023-09-29,32.438159,2023-09
4,2023-09,BBVA Colombia,5,Mayor a 25 SMLMV y menor o igual a 100 SMLMV,18196.74,616,299,2023-09-29,2023-09-29,29.540162,2023-09


,mes,mes_periodo,rango_orden,rango_monto,suma_tasa_credito,total_creditos,registros_fuente,fecha_minima,fecha_maxima,bancos,tasa_ponderada
0,2023-09,2023-09,1,Menor o igual a 3 SMLMV,382452.40,10305,4292,2023-09-29,2023-09-29,20,37.113285
1,2023-09,2023-09,2,Mayor a 3 SMLMV y menor o igual a 6 SMLMV,316865.57,8826,3463,2023-09-29,2023-09-29,19,35.901379
2,2023-09,2023-09,3,Mayor a 6 SMLMV y menor o igual a 12 SMLMV,299312.13,8729,3324,2023-09-29,2023-09-29,20,34.289395
3,2023-09,2023-09,4,Mayor a 12 SMLMV y menor o igual a 25 SMLMV,202815.06,6331,2623,2023-09-29,2023-09-29,20,32.035233
4,2023-09,2023-09,5,Mayor a 25 SMLMV y menor o igual a 100 SMLMV,258129.93,7720,2232,2023-09-29,2023-09-29,18,33.436519


## 3. Target T+3 y aptitud de series

El objetivo se define como la tasa ponderada tres meses después dentro de la misma serie. La aptitud se calcula para cada nivel jerárquico porque no todas las combinaciones tienen suficiente historia para un modelo granular.

El target es la tasa ponderada de 3 meses después. En este caso si estás en Febrero, el target es la tasa de Mayo.

La aptitud determina si una serie tiene suficiente calidad para modelar. Una serie es apta si cumple todo esto:

Al menos 18 meses de historia,
Al menos 12 observaciones del target disponibles,
Al menos 100 créditos en total,
Al menos 80% de cobertura de meses (sin huecos grandes),
Esto se calcula para los 4 niveles jerárquicos: banco+rango, banco, rango y total del sistema.

In [11]:
def preparar_nivel(df: pd.DataFrame, nivel: str, columnas_serie: list[str]) -> pd.DataFrame:
    base = df.copy()
    base["nivel"] = nivel
    base["serie_id"] = base[columnas_serie].astype(str).agg(" | ".join, axis=1)
    base["mes_periodo"] = pd.PeriodIndex(base["mes"], freq="M")
    base = base.sort_values(["serie_id", "mes_periodo"]).reset_index(drop=True)
    base["columnas_serie"] = ",".join(columnas_serie)
    return base


niveles = {
    "banco_rango": preparar_nivel(banco_rango_mes, "banco_rango", ["banco", "rango_monto"]),
    "banco": preparar_nivel(banco_mes, "banco", ["banco"]),
    "rango": preparar_nivel(rango_mes, "rango", ["rango_monto"]),
    "total": preparar_nivel(total_mes, "total", ["mes"]),
}
# Para total, la serie es única; se corrige después de preparar_nivel.
niveles["total"]["serie_id"] = "total_sistema"
niveles["total"]["columnas_serie"] = "total"


def agregar_target_t_plus_h(df: pd.DataFrame, horizonte: int) -> pd.DataFrame:
    data = df.sort_values(["serie_id", "mes_periodo"]).copy()
    grupo = data.groupby("serie_id", dropna=False)
    data[f"target_tasa_t{horizonte}"] = grupo["tasa_ponderada"].shift(-horizonte)
    data[f"target_mes_t{horizonte}"] = data["mes_periodo"] + horizonte
    data[f"target_creditos_t{horizonte}"] = grupo["total_creditos"].shift(-horizonte)
    data[f"target_mes_t{horizonte}"] = data[f"target_mes_t{horizonte}"].astype(str)
    return data


def resumir_aptitud(df: pd.DataFrame, horizonte: int) -> pd.DataFrame:
    target = f"target_tasa_t{horizonte}"
    resumen = (
        df.groupby(["nivel", "serie_id"], dropna=False)
        .agg(
            primer_mes=("mes_periodo", "min"),
            ultimo_mes=("mes_periodo", "max"),
            meses_observados=("mes", "nunique"),
            observaciones_modelables=(target, lambda serie: int(serie.notna().sum())),
            creditos_total=("total_creditos", "sum"),
            creditos_mediana=("total_creditos", "median"),
            tasa_minima=("tasa_ponderada", "min"),
            tasa_maxima=("tasa_ponderada", "max"),
            tasa_std=("tasa_ponderada", "std"),
        )
        .reset_index()
    )
    resumen["meses_esperados"] = resumen.apply(
        lambda fila: (fila["ultimo_mes"] - fila["primer_mes"]).n + 1,
        axis=1,
    )
    resumen["cobertura_meses"] = resumen["meses_observados"] / resumen["meses_esperados"]
    resumen["apta_modelo"] = (
        (resumen["meses_observados"] >= CONFIG["min_meses_serie"])
        & (resumen["observaciones_modelables"] >= CONFIG["min_observaciones_target"])
        & (resumen["creditos_total"] >= CONFIG["min_creditos_serie"])
        & (resumen["cobertura_meses"] >= CONFIG["min_cobertura_meses"])
    )
    resumen["primer_mes"] = resumen["primer_mes"].astype(str)
    resumen["ultimo_mes"] = resumen["ultimo_mes"].astype(str)
    return resumen.sort_values(["nivel", "apta_modelo", "creditos_total"], ascending=[True, False, False]).reset_index(drop=True)


niveles_t3 = {nombre: agregar_target_t_plus_h(df, HORIZONTE_MESES) for nombre, df in niveles.items()}
series_aptitud_t3 = pd.concat(
    [resumir_aptitud(df, HORIZONTE_MESES) for df in niveles_t3.values()],
    ignore_index=True,
)

for nivel, df in niveles_t3.items():
    aptas = series_aptitud_t3.query("nivel == @nivel and apta_modelo").shape[0]
    total = series_aptitud_t3.query("nivel == @nivel").shape[0]
    print(f"{nivel:<12}: {aptas:>3} / {total:<3} series aptas")

display(series_aptitud_t3.head(40))

banco_rango :  98 / 143 series aptas
banco       :  20 / 25  series aptas
rango       :   6 / 6   series aptas
total       :   1 / 1   series aptas


,nivel,serie_id,primer_mes,ultimo_mes,meses_observados,observaciones_modelables,creditos_total,creditos_mediana,tasa_minima,tasa_maxima,tasa_std,meses_esperados,cobertura_meses,apta_modelo
0,banco_rango,Bancolombia | Menor o igual a 3 SMLMV,2023-09,2026-03,31,28,641685,13553.0,21.948839,36.685589,4.572850,31,1.0,True
1,banco_rango,Banco Davibank | Mayor a 100 SMLMV,2023-09,2026-03,31,28,574174,6168.0,20.267688,41.828934,5.907003,31,1.0,True
2,banco_rango,Banco de Bogotá | Menor o igual a 3 SMLMV,2023-09,2026-03,31,28,398373,12404.0,23.879415,37.255322,4.460378,31,1.0,True
3,banco_rango,Banco de Bogotá | Mayor a 3 SMLMV y menor o ig...,2023-09,2026-03,31,28,300010,9757.0,23.607385,36.538372,4.329408,31,1.0,True
4,banco_rango,Bancolombia | Mayor a 3 SMLMV y menor o igual ...,2023-09,2026-03,31,28,246880,8606.0,22.109839,32.950226,3.473849,31,1.0,True
5,banco_rango,Banco de Bogotá | Mayor a 6 SMLMV y menor o ig...,2023-09,2026-03,31,28,219346,6894.0,23.142858,35.332474,4.088111,31,1.0,True
6,banco_rango,Bancolombia | Mayor a 6 SMLMV y menor o igual ...,2023-09,2026-03,31,28,208780,6239.0,20.929514,31.455167,3.460970,31,1.0,True
7,banco_rango,Banco Davivienda | Mayor a 6 SMLMV y menor o i...,2023-09,2026-03,31,28,187464,5557.0,21.112794,32.476997,3.681897,31,1.0,True
8,banco_rango,Banco Falabella | Menor o igual a 3 SMLMV,2023-09,2026-03,31,28,186107,6145.0,23.070540,37.774047,4.703002,31,1.0,True
9,banco_rango,Bancolombia | Mayor a 12 SMLMV y menor o igual...,2023-09,2026-03,31,28,176153,5454.0,19.360994,30.524269,3.538235,31,1.0,True


## 4. Ingeniería de características

Las variables se calculan usando solo información disponible en `t` o meses anteriores. Los rezagos y medias móviles se calculan dentro de cada serie para evitar fuga de información entre bancos o rangos.

Construye las variables que van a entrar a los modelos. Todo se calcula usando solo información del pasado para evitar fuga de información:

Lags de tasa — tasa de hace 1, 2, 3, 4, 6 y 12 meses

Medias móviles — promedio y desviación estándar de los últimos 3 y 6 meses

Diferencias — cuánto cambió la tasa respecto al mes anterior y al de hace 3 meses

Variables de tiempo — mes del año, trimestre y tendencia temporal

Variables de volumen — logaritmo del número de créditos y registros

In [14]:
LAGS_TASA = [1, 2, 3, 4, 6, 12]
VENTANAS_MOVILES = [3, 6]
TARGET_COL = f"target_tasa_t{HORIZONTE_MESES}"


def construir_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.sort_values(["serie_id", "mes_periodo"]).copy()
    grupo = data.groupby("serie_id", dropna=False)

    for lag in LAGS_TASA:
        data[f"lag_tasa_{lag}"] = grupo["tasa_ponderada"].shift(lag)
        data[f"lag_creditos_{lag}"] = grupo["total_creditos"].shift(lag)

    for ventana in VENTANAS_MOVILES:
        data[f"roll_mean_tasa_{ventana}"] = grupo["tasa_ponderada"].transform(
            lambda serie: serie.shift(1).rolling(ventana, min_periods=2).mean()
        )
        data[f"roll_std_tasa_{ventana}"] = grupo["tasa_ponderada"].transform(
            lambda serie: serie.shift(1).rolling(ventana, min_periods=2).std()
        )
        data[f"roll_mean_creditos_{ventana}"] = grupo["total_creditos"].transform(
            lambda serie: serie.shift(1).rolling(ventana, min_periods=2).mean()
        )

    data["diff_tasa_1"] = data["tasa_ponderada"] - data["lag_tasa_1"]
    data["diff_tasa_3"] = data["tasa_ponderada"] - data["lag_tasa_3"]
    data["pct_creditos_1"] = (data["total_creditos"] - data["lag_creditos_1"]) / data["lag_creditos_1"].replace(0, np.nan)

    data["mes_num"] = data["mes_periodo"].dt.month
    data["trimestre"] = data["mes_periodo"].dt.quarter
    data["tendencia"] = (data["mes_periodo"] - data["mes_periodo"].min()).apply(lambda offset: offset.n)
    data["log_creditos"] = np.log1p(data["total_creditos"])
    data["log_registros_fuente"] = np.log1p(data["registros_fuente"])

    return data


features_por_nivel = {nombre: construir_features(df) for nombre, df in niveles_t3.items()}

COLUMNAS_EXCLUIR_MODELO = {
    "suma_tasa_credito", "fecha_minima", "fecha_maxima", "mes_periodo", "mes", "target_mes_t3",
    TARGET_COL, "target_creditos_t3", "columnas_serie"
}

COLUMNAS_CATEGORICAS_BASE = ["nivel", "serie_id", "banco", "rango_monto"]
COLUMNAS_NUMERICAS_BASE = []
for df in features_por_nivel.values():
    for columna in df.columns:
        if columna in COLUMNAS_EXCLUIR_MODELO:
            continue
        if pd.api.types.is_numeric_dtype(df[columna]) and columna not in COLUMNAS_NUMERICAS_BASE:
            COLUMNAS_NUMERICAS_BASE.append(columna)

print("Columnas numéricas candidatas:", COLUMNAS_NUMERICAS_BASE)
print("Columnas categóricas base:", COLUMNAS_CATEGORICAS_BASE)

display(features_por_nivel["banco_rango"].head())

Columnas numéricas candidatas: ['rango_orden', 'total_creditos', 'registros_fuente', 'tasa_ponderada', 'lag_tasa_1', 'lag_creditos_1', 'lag_tasa_2', 'lag_creditos_2', 'lag_tasa_3', 'lag_creditos_3', 'lag_tasa_4', 'lag_creditos_4', 'lag_tasa_6', 'lag_creditos_6', 'lag_tasa_12', 'lag_creditos_12', 'roll_mean_tasa_3', 'roll_std_tasa_3', 'roll_mean_creditos_3', 'roll_mean_tasa_6', 'roll_std_tasa_6', 'roll_mean_creditos_6', 'diff_tasa_1', 'diff_tasa_3', 'pct_creditos_1', 'mes_num', 'trimestre', 'tendencia', 'log_creditos', 'log_registros_fuente', 'bancos']
Columnas categóricas base: ['nivel', 'serie_id', 'banco', 'rango_monto']


,mes,banco,rango_orden,rango_monto,suma_tasa_credito,total_creditos,registros_fuente,fecha_minima,fecha_maxima,tasa_ponderada,mes_periodo,nivel,serie_id,columnas_serie,target_tasa_t3,target_mes_t3,target_creditos_t3,lag_tasa_1,lag_creditos_1,lag_tasa_2,lag_creditos_2,lag_tasa_3,lag_creditos_3,lag_tasa_4,lag_creditos_4,lag_tasa_6,lag_creditos_6,lag_tasa_12,lag_creditos_12,roll_mean_tasa_3,roll_std_tasa_3,roll_mean_creditos_3,roll_mean_tasa_6,roll_std_tasa_6,roll_mean_creditos_6,diff_tasa_1,diff_tasa_3,pct_creditos_1,mes_num,trimestre,tendencia,log_creditos,log_registros_fuente
0,2023-09,BBVA Colombia,6,Mayor a 100 SMLMV,585.39,23,21,2023-09-29,2023-09-29,25.451739,2023-09,banco_rango,BBVA Colombia | Mayor a 100 SMLMV,"banco,rango_monto",23.623864,2023-12,88.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9,3,0,3.178054,3.091042
1,2023-10,BBVA Colombia,6,Mayor a 100 SMLMV,1101.89,44,41,2023-10-06,2023-10-27,25.042955,2023-10,banco_rango,BBVA Colombia | Mayor a 100 SMLMV,"banco,rango_monto",19.785200,2024-01,25.0,25.451739,23.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.408785,NaN,0.913043,10,4,1,3.806662,3.737670
2,2023-11,BBVA Colombia,6,Mayor a 100 SMLMV,1241.86,54,51,2023-11-03,2023-11-24,22.997407,2023-11,banco_rango,BBVA Colombia | Mayor a 100 SMLMV,"banco,rango_monto",22.531071,2024-02,28.0,25.042955,44.0,25.451739,23.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25.247347,0.289054,33.500000,25.247347,0.289054,33.500000,-2.045547,NaN,0.227273,11,4,2,4.007333,3.951244
3,2023-12,BBVA Colombia,6,Mayor a 100 SMLMV,2078.90,88,78,2023-12-01,2023-12-29,23.623864,2023-12,banco_rango,BBVA Colombia | Mayor a 100 SMLMV,"banco,rango_monto",21.527955,2024-03,44.0,22.997407,54.0,25.042955,44.0,25.451739,23.0,NaN,NaN,NaN,NaN,NaN,NaN,24.497367,1.314985,40.333333,24.497367,1.314985,40.333333,0.626456,-1.827875,0.629630,12,4,3,4.488636,4.369448
4,2024-01,BBVA Colombia,6,Mayor a 100 SMLMV,494.63,25,24,2024-01-05,2024-01-26,19.785200,2024-01,banco_rango,BBVA Colombia | Mayor a 100 SMLMV,"banco,rango_monto",20.151556,2024-04,45.0,23.623864,88.0,22.997407,54.0,25.042955,44.0,25.451739,23.0,NaN,NaN,NaN,NaN,23.888075,1.048056,62.000000,24.278991,1.159113,52.250000,-3.838664,-5.257755,-0.715909,1,1,4,3.258097,3.218876


## 5. Validación temporal y baselines

La evaluación separa los últimos meses con target disponible como test y los meses previos como validación. Los baselines son obligatorios: un modelo optimizado solo se acepta si aporta frente a estos puntos de comparación.

Define 4 baselines obligatorios de comparación:

In [17]:
def smape(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denom != 0
    if not mask.any():
        return np.nan
    return float(np.mean(np.abs(y_true[mask] - y_pred[mask]) / denom[mask]) * 100)


def calcular_metricas(y_true, y_pred) -> dict:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "r2": float(r2_score(y_true, y_pred)) if len(np.unique(y_true)) > 1 else np.nan,
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "smape": smape(y_true, y_pred),
        "n": int(len(y_true)),
    }


def preparar_particiones_temporales(df: pd.DataFrame) -> dict:
    data = df[df[TARGET_COL].notna()].copy()
    meses = sorted(data["mes_periodo"].unique())
    if len(meses) < CONFIG["valid_meses"] + CONFIG["test_meses"] + 6:
        raise ValueError("No hay suficientes meses para validación temporal robusta")

    test_meses = meses[-CONFIG["test_meses"]:]
    valid_meses = meses[-(CONFIG["test_meses"] + CONFIG["valid_meses"]):-CONFIG["test_meses"]]
    train_meses = meses[:-(CONFIG["test_meses"] + CONFIG["valid_meses"])]

    return {
        "train": data[data["mes_periodo"].isin(train_meses)].copy(),
        "valid": data[data["mes_periodo"].isin(valid_meses)].copy(),
        "test": data[data["mes_periodo"].isin(test_meses)].copy(),
        "train_valid": data[~data["mes_periodo"].isin(test_meses)].copy(),
        "meses_train": train_meses,
        "meses_valid": valid_meses,
        "meses_test": test_meses,
    }


def evaluar_baselines(df: pd.DataFrame, nivel: str) -> pd.DataFrame:
    data = df[df[TARGET_COL].notna()].copy()
    particiones = preparar_particiones_temporales(data)
    test = particiones["test"].copy()

    candidatos = {
        "baseline_naive_lag1": test["lag_tasa_1"],
        "baseline_media_movil_3": test["roll_mean_tasa_3"],
        "baseline_media_movil_6": test["roll_mean_tasa_6"],
        "baseline_tasa_actual": test["tasa_ponderada"],
    }

    filas = []
    for modelo, pred in candidatos.items():
        eval_df = pd.DataFrame({"y": test[TARGET_COL], "pred": pred}).dropna()
        if eval_df.empty:
            continue
        metricas = calcular_metricas(eval_df["y"], eval_df["pred"])
        filas.append({"nivel": nivel, "modelo": modelo, "split": "test", **metricas})

    return pd.DataFrame(filas)


metricas_baseline = pd.concat(
    [evaluar_baselines(df, nivel) for nivel, df in features_por_nivel.items()],
    ignore_index=True,
)

display(metricas_baseline.sort_values(["nivel", "r2"], ascending=[True, False]))

,nivel,modelo,split,r2,mae,rmse,smape,n
7,banco,baseline_tasa_actual,test,0.891665,0.900043,1.279304,5.317198,126
6,banco,baseline_media_movil_6,test,0.875240,1.036163,1.372860,5.730045,126
5,banco,baseline_media_movil_3,test,0.867976,1.049496,1.412259,5.819501,126
4,banco,baseline_naive_lag1,test,0.856145,1.078705,1.474180,6.225560,126
2,banco_rango,baseline_media_movil_6,test,0.811596,1.093558,1.518770,5.913753,661
3,banco_rango,baseline_tasa_actual,test,0.810410,0.934599,1.523545,5.123787,661
1,banco_rango,baseline_media_movil_3,test,0.800479,1.086276,1.562939,5.864561,661
0,banco_rango,baseline_naive_lag1,test,0.778198,1.089746,1.647898,5.812224,661
11,rango,baseline_tasa_actual,test,0.784475,0.535604,0.700076,2.497880,36
8,rango,baseline_naive_lag1,test,0.719564,0.641270,0.798570,2.962582,36


## 6. Optimización bayesiana con Optuna

Cada nivel jerárquico se entrena como un modelo supervisado con validación temporal. Optuna maximiza `R2` en validación; después el mejor modelo se reentrena con `train + valid` y se evalúa en `test`.

Esta es la parte central. Optuna es una librería de optimización que encuentra los mejores hiperparámetros de forma inteligente — no prueba combinaciones al azar sino que aprende qué rangos funcionan mejor con cada prueba.

In [20]:
import os
print("Cores lógicos:", os.cpu_count())

Cores lógicos: 8


In [22]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "8"

CONFIG["n_jobs_modelos_arboles"] = 6

In [26]:
def columnas_modelo_disponibles(df: pd.DataFrame) -> tuple[list[str], list[str]]:
    numericas = [col for col in COLUMNAS_NUMERICAS_BASE if col in df.columns]
    categoricas = [col for col in COLUMNAS_CATEGORICAS_BASE if col in df.columns]
    return numericas, categoricas


def crear_preprocesador(numericas: list[str], categoricas: list[str]) -> ColumnTransformer:
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("num", numeric_pipe, numericas),
        ("cat", categorical_pipe, categoricas),
    ], remainder="drop")


def sugerir_modelo(trial: optuna.Trial, familia: str):
    if familia == "ridge":
        return Ridge(alpha=trial.suggest_float("alpha", 1e-4, 1e3, log=True), random_state=RANDOM_STATE)
    if familia == "elasticnet":
        return ElasticNet(
            alpha=trial.suggest_float("alpha", 1e-4, 10.0, log=True),
            l1_ratio=trial.suggest_float("l1_ratio", 0.05, 0.95),
            max_iter=20000,
            random_state=RANDOM_STATE,
        )
    if familia == "random_forest":
        return RandomForestRegressor(
            n_estimators=trial.suggest_int("n_estimators", 200, 900),
            max_depth=trial.suggest_int("max_depth", 2, 16),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 12),
            min_samples_split=trial.suggest_int("min_samples_split", 2, 24),
            max_features=trial.suggest_float("max_features", 0.35, 1.0),
            random_state=RANDOM_STATE,
            n_jobs=CONFIG["n_jobs_modelos_arboles"],
        )
    if familia == "extra_trees":
        return ExtraTreesRegressor(
            n_estimators=trial.suggest_int("n_estimators", 200, 1000),
            max_depth=trial.suggest_int("max_depth", 2, 18),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 12),
            min_samples_split=trial.suggest_int("min_samples_split", 2, 24),
            max_features=trial.suggest_float("max_features", 0.35, 1.0),
            random_state=RANDOM_STATE,
            n_jobs=CONFIG["n_jobs_modelos_arboles"],
        )
    if familia == "hist_gradient_boosting":
        return HistGradientBoostingRegressor(
            learning_rate=trial.suggest_float("learning_rate", 0.005, 0.25, log=True),
            max_iter=trial.suggest_int("max_iter", 100, 700),
            max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 8, 64),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 5, 50),
            l2_regularization=trial.suggest_float("l2_regularization", 1e-6, 10.0, log=True),
            random_state=RANDOM_STATE,
        )
    if familia == "xgboost" and XGBOOST_DISPONIBLE:
        return XGBRegressor(
            n_estimators=trial.suggest_int("n_estimators", 150, 900),
            max_depth=trial.suggest_int("max_depth", 2, 10),
            learning_rate=trial.suggest_float("learning_rate", 0.005, 0.25, log=True),
            subsample=trial.suggest_float("subsample", 0.55, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.55, 1.0),
            min_child_weight=trial.suggest_float("min_child_weight", 1.0, 20.0),
            reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=CONFIG["n_jobs_modelos_arboles"],
        )
    raise ValueError(f"Familia no soportada o no disponible: {familia}")


def entrenar_familia_optuna(df: pd.DataFrame, nivel: str, familia: str) -> dict:
    particiones = preparar_particiones_temporales(df)
    train, valid, test, train_valid = (
        particiones["train"], particiones["valid"], particiones["test"], particiones["train_valid"]
    )
    numericas, categoricas = columnas_modelo_disponibles(df)
    feature_cols = numericas + categoricas

    X_train, y_train = train[feature_cols], train[TARGET_COL]
    X_valid, y_valid = valid[feature_cols], valid[TARGET_COL]
    X_test, y_test = test[feature_cols], test[TARGET_COL]

    def objective(trial: optuna.Trial) -> float:
        modelo = sugerir_modelo(trial, familia)
        pipe = Pipeline([
            ("prep", crear_preprocesador(numericas, categoricas)),
            ("model", modelo),
        ])
        pipe.fit(X_train, y_train)
        pred_valid = pipe.predict(X_valid)
        metricas_valid = calcular_metricas(y_valid, pred_valid)
        trial.set_user_attr("mae_valid", metricas_valid["mae"])
        trial.set_user_attr("rmse_valid", metricas_valid["rmse"])
        trial.set_user_attr("smape_valid", metricas_valid["smape"])
        return metricas_valid["r2"] if np.isfinite(metricas_valid["r2"]) else -999.0

    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=RANDOM_STATE),
        study_name=f"{nivel}_{familia}_t{HORIZONTE_MESES}",
    )
    study.optimize(
        objective,
        n_trials=CONFIG["n_trials_intensivo"],
        timeout=CONFIG["timeout_estudio_segundos"],
        show_progress_bar=False,
    )

    mejor_modelo = sugerir_modelo(study.best_trial, familia)
    mejor_pipe = Pipeline([
        ("prep", crear_preprocesador(numericas, categoricas)),
        ("model", mejor_modelo),
    ])
    mejor_pipe.fit(train_valid[feature_cols], train_valid[TARGET_COL])
    pred_test = mejor_pipe.predict(X_test)
    metricas_test = calcular_metricas(y_test, pred_test)

    predicciones = test[["nivel", "serie_id", "mes", TARGET_COL]].copy()
    predicciones["modelo"] = familia
    predicciones["prediccion"] = pred_test
    predicciones["error"] = predicciones[TARGET_COL] - predicciones["prediccion"]

    trials = study.trials_dataframe(attrs=("number", "value", "params", "user_attrs", "state"))
    trials["nivel"] = nivel
    trials["familia"] = familia

    resultado = {
        "nivel": nivel,
        "modelo": familia,
        "pipeline": mejor_pipe,
        "best_params": study.best_params,
        "best_value_valid_r2": study.best_value,
        "metricas_test": {"nivel": nivel, "modelo": familia, "split": "test", **metricas_test},
        "predicciones": predicciones,
        "trials": trials,
        "feature_cols": feature_cols,
    }
    return resultado


familias_modelos = ["ridge", "elasticnet", "random_forest", "extra_trees", "hist_gradient_boosting"]
if XGBOOST_DISPONIBLE:
    familias_modelos.append("xgboost")

resultados_optuna = []
for nivel, df in features_por_nivel.items():
    print(f"\nNivel: {nivel}")
    for familia in familias_modelos:
        print(f"  Optimizando {familia}...")
        resultado = entrenar_familia_optuna(df, nivel, familia)
        resultados_optuna.append(resultado)
        print(
            f"    R2 valid={resultado['best_value_valid_r2']:.4f} | "
            f"R2 test={resultado['metricas_test']['r2']:.4f} | "
            f"MAE test={resultado['metricas_test']['mae']:.4f}"
        )

metricas_optuna = pd.DataFrame([res["metricas_test"] for res in resultados_optuna])
optuna_trials_t3 = pd.concat([res["trials"] for res in resultados_optuna], ignore_index=True)
predicciones_optuna_test = pd.concat([res["predicciones"] for res in resultados_optuna], ignore_index=True)
modelos_entrenados = {(res["nivel"], res["modelo"]): res["pipeline"] for res in resultados_optuna}

metricas_modelos_t3 = pd.concat([metricas_baseline, metricas_optuna], ignore_index=True)
display(metricas_modelos_t3.sort_values(["nivel", "r2"], ascending=[True, False]))


Nivel: banco_rango
  Optimizando ridge...
    R2 valid=0.7284 | R2 test=0.5079 | MAE test=2.0080
  Optimizando elasticnet...
    R2 valid=0.7824 | R2 test=0.7431 | MAE test=1.3438
  Optimizando random_forest...
    R2 valid=0.8128 | R2 test=0.7730 | MAE test=1.3060
  Optimizando extra_trees...
    R2 valid=0.7437 | R2 test=0.7728 | MAE test=1.2850
  Optimizando hist_gradient_boosting...
    R2 valid=0.8106 | R2 test=0.7741 | MAE test=1.3057
  Optimizando xgboost...
    R2 valid=0.8129 | R2 test=0.7866 | MAE test=1.3049

Nivel: banco
  Optimizando ridge...
    R2 valid=0.8592 | R2 test=0.5607 | MAE test=2.1683
  Optimizando elasticnet...
    R2 valid=0.8548 | R2 test=0.5618 | MAE test=2.1678
  Optimizando random_forest...
    R2 valid=0.8427 | R2 test=0.8084 | MAE test=1.3871
  Optimizando extra_trees...
    R2 valid=0.8563 | R2 test=0.8202 | MAE test=1.3477
  Optimizando hist_gradient_boosting...
    R2 valid=0.6925 | R2 test=0.6734 | MAE test=1.5460
  Optimizando xgboost...
    R2 va

,nivel,modelo,split,r2,mae,rmse,smape,n
7,banco,baseline_tasa_actual,test,0.891665,0.900043,1.279304,5.317198,126
6,banco,baseline_media_movil_6,test,0.875240,1.036163,1.372860,5.730045,126
5,banco,baseline_media_movil_3,test,0.867976,1.049496,1.412259,5.819501,126
4,banco,baseline_naive_lag1,test,0.856145,1.078705,1.474180,6.225560,126
25,banco,extra_trees,test,0.820217,1.347665,1.648021,7.134408,126
24,banco,random_forest,test,0.808369,1.387094,1.701458,7.347359,126
27,banco,xgboost,test,0.694122,1.646178,2.149624,9.400378,126
26,banco,hist_gradient_boosting,test,0.673419,1.546010,2.221181,9.118386,126
23,banco,elasticnet,test,0.561753,2.167791,2.573049,11.973467,126
22,banco,ridge,test,0.560651,2.168314,2.576282,11.989442,126


## 7. Selección de modelo y fallback jerárquico

Se escoge el mejor modelo por nivel según `R2` en test, comparando modelos optimizados y baselines. Luego cada combinación `banco + rango` usa el nivel más granular que tenga aptitud suficiente.

In [29]:
def seleccionar_mejor_modelo_por_nivel(metricas: pd.DataFrame, tolerancia_error: float = 0.05) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Selecciona modelos priorizando R2 sin aceptar deterioros relevantes en MAE/RMSE."""
    seleccionados = []
    auditoria = []

    for nivel, grupo in metricas.groupby("nivel", sort=True):
        grupo = grupo.copy()
        grupo["r2_orden"] = grupo["r2"].fillna(-999)
        baselines = grupo[grupo["modelo"].str.startswith("baseline")]
        baseline_ref = baselines.sort_values(
            ["r2_orden", "mae", "rmse"], ascending=[False, True, True]
        ).iloc[0]

        grupo["cumple_r2"] = grupo["r2_orden"] >= baseline_ref["r2_orden"]
        grupo["cumple_mae"] = grupo["mae"] <= baseline_ref["mae"] * (1 + tolerancia_error)
        grupo["cumple_rmse"] = grupo["rmse"] <= baseline_ref["rmse"] * (1 + tolerancia_error)
        grupo["es_aceptable"] = grupo["cumple_r2"] & grupo["cumple_mae"] & grupo["cumple_rmse"]

        aceptables = grupo[grupo["es_aceptable"]]
        if aceptables.empty:
            elegido = baseline_ref.copy()
            motivo = "baseline_ref: ningun modelo mejora R2 sin deteriorar MAE/RMSE"
        else:
            elegido = aceptables.sort_values(
                ["r2_orden", "mae", "rmse"], ascending=[False, True, True]
            ).iloc[0].copy()
            motivo = "modelo aceptado: R2 >= baseline y MAE/RMSE dentro de tolerancia"

        auditoria.append(
            {
                "nivel": nivel,
                "baseline_referencia": baseline_ref["modelo"],
                "r2_baseline": baseline_ref["r2"],
                "mae_baseline": baseline_ref["mae"],
                "rmse_baseline": baseline_ref["rmse"],
                "modelo_recomendado": elegido["modelo"],
                "r2_recomendado": elegido["r2"],
                "mae_recomendado": elegido["mae"],
                "rmse_recomendado": elegido["rmse"],
                "motivo": motivo,
            }
        )
        seleccionados.append(elegido)

    mejores = pd.DataFrame(seleccionados).drop(
        columns=["r2_orden", "cumple_r2", "cumple_mae", "cumple_rmse", "es_aceptable"],
        errors="ignore",
    )
    return mejores, pd.DataFrame(auditoria)


mejores_modelos_t3, criterio_seleccion_modelos_t3 = seleccionar_mejor_modelo_por_nivel(metricas_modelos_t3)
MEJOR_MODELO_NIVEL = dict(zip(mejores_modelos_t3["nivel"], mejores_modelos_t3["modelo"]))


def serie_apta(nivel: str, serie_id: str) -> bool:
    fila = series_aptitud_t3[(series_aptitud_t3["nivel"] == nivel) & (series_aptitud_t3["serie_id"] == serie_id)]
    return bool(not fila.empty and fila.iloc[0]["apta_modelo"])


def obtener_ultima_fila(nivel: str, filtro: dict | None = None) -> pd.Series:
    df = features_por_nivel[nivel].copy()
    if filtro:
        for columna, valor in filtro.items():
            df = df[df[columna] == valor]
    if df.empty:
        raise ValueError(f"No hay filas para nivel={nivel}, filtro={filtro}")
    return df.sort_values("mes_periodo").iloc[-1]


def predecir_fila(row: pd.Series, nivel: str, modelo_nombre: str) -> float:
    if modelo_nombre == "baseline_naive_lag1":
        return float(row["lag_tasa_1"])
    if modelo_nombre == "baseline_media_movil_3":
        return float(row["roll_mean_tasa_3"])
    if modelo_nombre == "baseline_media_movil_6":
        return float(row["roll_mean_tasa_6"])
    if modelo_nombre == "baseline_tasa_actual":
        return float(row["tasa_ponderada"])

    pipeline = modelos_entrenados[(nivel, modelo_nombre)]
    feature_cols = [col for col in pipeline.feature_names_in_] if hasattr(pipeline, "feature_names_in_") else None
    if feature_cols is None:
        numericas, categoricas = columnas_modelo_disponibles(features_por_nivel[nivel])
        feature_cols = numericas + categoricas
    X = pd.DataFrame([row[feature_cols].to_dict()])
    return float(pipeline.predict(X)[0])


def decidir_fallback(row_banco_rango: pd.Series) -> dict:
    banco = row_banco_rango["banco"]
    rango = row_banco_rango["rango_monto"]
    serie_br = row_banco_rango["serie_id"]

    if serie_apta("banco_rango", serie_br):
        nivel = "banco_rango"
        filtro = {"banco": banco, "rango_monto": rango}
    elif serie_apta("banco", banco):
        nivel = "banco"
        filtro = {"banco": banco}
    elif serie_apta("rango", rango):
        nivel = "rango"
        filtro = {"rango_monto": rango}
    else:
        nivel = "total"
        filtro = None

    fila_pred = obtener_ultima_fila(nivel, filtro)
    modelo = MEJOR_MODELO_NIVEL[nivel]
    prediccion = predecir_fila(fila_pred, nivel, modelo)
    mes_base = fila_pred["mes_periodo"]
    mes_predicho = mes_base + HORIZONTE_MESES

    return {
        "banco": banco,
        "rango_orden": row_banco_rango.get("rango_orden", np.nan),
        "rango_monto": rango,
        "serie_banco_rango": serie_br,
        "nivel_usado": nivel,
        "serie_usada": fila_pred["serie_id"],
        "modelo_usado": modelo,
        "mes_base": str(mes_base),
        "mes_predicho": str(mes_predicho),
        "tasa_base": float(row_banco_rango["tasa_ponderada"]),
        "prediccion_tasa_t3": prediccion,
        "total_creditos_base": float(row_banco_rango["total_creditos"]),
    }


ultimo_mes_banco_rango = features_por_nivel["banco_rango"]["mes_periodo"].max()
filas_actuales_banco_rango = features_por_nivel["banco_rango"].query("mes_periodo == @ultimo_mes_banco_rango").copy()

fallback_decisions_t3 = pd.DataFrame([decidir_fallback(fila) for _, fila in filas_actuales_banco_rango.iterrows()])
predicciones_t3 = fallback_decisions_t3.copy()

print("Mejor modelo por nivel:")
display(mejores_modelos_t3)
print("Decisiones de fallback:")
display(fallback_decisions_t3["nivel_usado"].value_counts().to_frame("casos"))
display(predicciones_t3.head(30))

Mejor modelo por nivel:


,nivel,modelo,split,r2,mae,rmse,smape,n
7,banco,baseline_tasa_actual,test,0.891665,0.900043,1.279304,5.317198,126
2,banco_rango,baseline_media_movil_6,test,0.811596,1.093558,1.518770,5.913753,661
11,rango,baseline_tasa_actual,test,0.784475,0.535604,0.700076,2.497880,36
15,total,baseline_tasa_actual,test,-0.239994,0.541163,0.616297,2.404778,6


Decisiones de fallback:


,casos
nivel_usado,
banco_rango,95
banco,17
rango,7


,banco,rango_orden,rango_monto,serie_banco_rango,nivel_usado,serie_usada,modelo_usado,mes_base,mes_predicho,tasa_base,prediccion_tasa_t3,total_creditos_base
0,BBVA Colombia,6,Mayor a 100 SMLMV,BBVA Colombia | Mayor a 100 SMLMV,banco_rango,BBVA Colombia | Mayor a 100 SMLMV,baseline_media_movil_6,2026-03,2026-06,22.468750,20.298648,8.0
1,BBVA Colombia,4,Mayor a 12 SMLMV y menor o igual a 25 SMLMV,BBVA Colombia | Mayor a 12 SMLMV y menor o igu...,banco_rango,BBVA Colombia | Mayor a 12 SMLMV y menor o igu...,baseline_media_movil_6,2026-03,2026-06,24.250097,22.789439,2370.0
2,BBVA Colombia,5,Mayor a 25 SMLMV y menor o igual a 100 SMLMV,BBVA Colombia | Mayor a 25 SMLMV y menor o igu...,banco_rango,BBVA Colombia | Mayor a 25 SMLMV y menor o igu...,baseline_media_movil_6,2026-03,2026-06,23.497652,21.592266,1712.0
3,BBVA Colombia,2,Mayor a 3 SMLMV y menor o igual a 6 SMLMV,BBVA Colombia | Mayor a 3 SMLMV y menor o igua...,banco_rango,BBVA Colombia | Mayor a 3 SMLMV y menor o igua...,baseline_media_movil_6,2026-03,2026-06,24.957817,23.671722,3151.0
4,BBVA Colombia,3,Mayor a 6 SMLMV y menor o igual a 12 SMLMV,BBVA Colombia | Mayor a 6 SMLMV y menor o igua...,banco_rango,BBVA Colombia | Mayor a 6 SMLMV y menor o igua...,baseline_media_movil_6,2026-03,2026-06,24.863980,23.230776,4121.0
5,BBVA Colombia,1,Menor o igual a 3 SMLMV,BBVA Colombia | Menor o igual a 3 SMLMV,banco_rango,BBVA Colombia | Menor o igual a 3 SMLMV,baseline_media_movil_6,2026-03,2026-06,25.201112,24.213574,5101.0
6,Banco AV Villas,6,Mayor a 100 SMLMV,Banco AV Villas | Mayor a 100 SMLMV,banco_rango,Banco AV Villas | Mayor a 100 SMLMV,baseline_media_movil_6,2026-03,2026-06,18.805694,17.003504,72.0
7,Banco AV Villas,4,Mayor a 12 SMLMV y menor o igual a 25 SMLMV,Banco AV Villas | Mayor a 12 SMLMV y menor o i...,banco_rango,Banco AV Villas | Mayor a 12 SMLMV y menor o i...,baseline_media_movil_6,2026-03,2026-06,20.879879,18.755074,660.0
8,Banco AV Villas,5,Mayor a 25 SMLMV y menor o igual a 100 SMLMV,Banco AV Villas | Mayor a 25 SMLMV y menor o i...,banco_rango,Banco AV Villas | Mayor a 25 SMLMV y menor o i...,baseline_media_movil_6,2026-03,2026-06,19.190701,17.386141,827.0
9,Banco AV Villas,2,Mayor a 3 SMLMV y menor o igual a 6 SMLMV,Banco AV Villas | Mayor a 3 SMLMV y menor o ig...,banco_rango,Banco AV Villas | Mayor a 3 SMLMV y menor o ig...,baseline_media_movil_6,2026-03,2026-06,22.885588,21.429310,791.0


## 8. Exportación de resultados

Se exportan predicciones, métricas, decisiones de fallback, trials de Optuna y modelos entrenados. La interpretación narrativa del proyecto (problema, métodos, resultados y lectura por sección) está en **`README.md`** y en **`docs/INFORME_MODELADO.md`**.

In [32]:
resumen_ejecutivo_modelado_t3 = pd.DataFrame([
    {"indicador": "horizonte_meses", "valor": HORIZONTE_MESES},
    {"indicador": "niveles_modelados", "valor": ", ".join(sorted(features_por_nivel.keys()))},
    {"indicador": "n_trials_por_estudio", "valor": CONFIG["n_trials_intensivo"]},
    {"indicador": "series_banco_rango", "valor": int(series_aptitud_t3.query("nivel == 'banco_rango'").shape[0])},
    {"indicador": "series_banco_rango_aptas", "valor": int(series_aptitud_t3.query("nivel == 'banco_rango' and apta_modelo").shape[0])},
    {"indicador": "predicciones_finales", "valor": len(predicciones_t3)},
])

if "criterio_seleccion_modelos_t3" not in globals():
    _ruta_criterio = OUTPUT_MODELOS / "criterio_seleccion_modelos_t3.csv"
    if _ruta_criterio.exists():
        criterio_seleccion_modelos_t3 = pd.read_csv(_ruta_criterio)
    else:
        _, criterio_seleccion_modelos_t3 = seleccionar_mejor_modelo_por_nivel(metricas_modelos_t3)

EXPORTACIONES_MODELO = {
    "predicciones_t3.csv": predicciones_t3,
    "metricas_modelos_t3.csv": metricas_modelos_t3,
    "mejores_modelos_t3.csv": mejores_modelos_t3,
    "criterio_seleccion_modelos_t3.csv": criterio_seleccion_modelos_t3,
    "optuna_trials_t3.csv": optuna_trials_t3,
    "series_aptitud_t3.csv": series_aptitud_t3,
    "fallback_decisions_t3.csv": fallback_decisions_t3,
    "resumen_ejecutivo_modelado_t3.csv": resumen_ejecutivo_modelado_t3,
    "predicciones_optuna_test_t3.csv": predicciones_optuna_test,
}

for nombre_archivo, df in EXPORTACIONES_MODELO.items():
    ruta = OUTPUT_MODELOS / nombre_archivo
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"Guardado: {ruta} ({len(df):,} filas)")

modelos_dir = OUTPUT_MODELOS / "modelos"
modelos_dir.mkdir(exist_ok=True)
for (nivel, modelo), pipeline in modelos_entrenados.items():
    ruta_modelo = modelos_dir / f"modelo_{nivel}_{modelo}_t{HORIZONTE_MESES}.joblib"
    joblib.dump(pipeline, ruta_modelo)

with open(OUTPUT_MODELOS / "config_modelado_t3.json", "w", encoding="utf-8") as archivo:
    json.dump(CONFIG | {"horizonte_meses": HORIZONTE_MESES}, archivo, ensure_ascii=False, indent=2)

print(f"Modelos guardados en: {modelos_dir}")
display(resumen_ejecutivo_modelado_t3)

Guardado: outputs_modelado\predicciones_t3.csv (119 filas)
Guardado: outputs_modelado\metricas_modelos_t3.csv (40 filas)
Guardado: outputs_modelado\mejores_modelos_t3.csv (4 filas)
Guardado: outputs_modelado\criterio_seleccion_modelos_t3.csv (4 filas)
Guardado: outputs_modelado\optuna_trials_t3.csv (1,920 filas)
Guardado: outputs_modelado\series_aptitud_t3.csv (175 filas)
Guardado: outputs_modelado\fallback_decisions_t3.csv (119 filas)
Guardado: outputs_modelado\resumen_ejecutivo_modelado_t3.csv (6 filas)
Guardado: outputs_modelado\predicciones_optuna_test_t3.csv (4,974 filas)
Modelos guardados en: outputs_modelado\modelos


,indicador,valor
0,horizonte_meses,3
1,niveles_modelados,"banco, banco_rango, rango, total"
2,n_trials_por_estudio,80
3,series_banco_rango,143
4,series_banco_rango_aptas,98
5,predicciones_finales,119


## 9. Documentación del proyecto

La interpretación de cada sección del notebook y del trabajo completo está en Markdown (no en código automático):

| Documento | Contenido |
|-----------|-----------|
| [`README.md`](../README.md) | Problema, objetivos, arquitectura, resultados y conclusiones |
| [`docs/INFORME_ETL.md`](../docs/INFORME_ETL.md) | Lectura sección por sección del notebook ETL |
| [`docs/INFORME_MODELADO.md`](../docs/INFORME_MODELADO.md) | Lectura sección por sección del notebook de modelado |

Archivos de datos generados por la celda 8: `outputs_modelado/predicciones_t3.csv`, `metricas_modelos_t3.csv`, `fallback_decisions_t3.csv`, entre otros.

In [5]:
import pandas as pd

# Importar el CSV
df = pd.read_csv("predicciones_t3.csv")  # cambia por el nombre de tu archivo

# Crear columnas nuevas con formato porcentaje
df['tasa_base_pct'] = df['tasa_base'].apply(lambda x: f"{x:.2f}".replace(".", ",") + "%")
df['prediccion_tasa_t3_pct'] = df['prediccion_tasa_t3'].apply(lambda x: f"{x:.2f}".replace(".", ",") + "%")

# Guardar en archivo nuevo
df.to_csv("archivo_con_porcentajes.csv", index=False)

print("✅ Archivo guardado exitosamente")
print(df[['tasa_base', 'tasa_base_pct', 'prediccion_tasa_t3', 'prediccion_tasa_t3_pct']].head())

✅ Archivo guardado exitosamente
   tasa_base tasa_base_pct  prediccion_tasa_t3 prediccion_tasa_t3_pct
0  19.889167        19,89%           19.889167                 19,89%
1  23.406055        23,41%           23.406055                 23,41%
2  22.572867        22,57%           22.572867                 22,57%
3  24.538290        24,54%           24.538290                 24,54%
4  23.870644        23,87%           23.870644                 23,87%
